---
# 1. Setup and Installation

This section installs all required packages and imports necessary libraries for the assignment.

## 1.1 Install Required Packages

Install the following packages:
- **transformers**: For pre-trained models (FLAN-T5, Llama)
- **datasets**: For loading XSum dataset from Hugging Face
- **accelerate**: For distributed training and mixed precision
- **evaluate**: For evaluation metrics
- **rouge_score, bert_score, sacrebleu**: For text generation metrics
- **sentencepiece, protobuf**: For tokenization
- **summac**: For factual consistency evaluation
- **huggingface_hub**: For uploading models
- **peft**: For LoRA (Parameter-Efficient Fine-Tuning)

In [ ]:
# Install core packages with compatible versions
print("📦 Installing packages... This may take 2-3 minutes.")
print("⚠️  You may see some dependency warnings - these are usually safe to ignore.\n")

# Strategy: Install a compatible ecosystem that works together
# Using transformers 4.35.2 which is already present and compatible

# First, uninstall conflicting packages
!pip uninstall -y -q peft summac bitsandbytes 2>/dev/null || true

# Install transformers and peft with compatible versions
!pip install -q transformers==4.35.2 peft==0.6.2

# Note: We're using Kaggle's pre-installed datasets library
# Skip bitsandbytes - it has CUDA 12.4 compatibility issues on Kaggle
# We'll use regular precision instead of quantization (still works fine on T4)

# Install core evaluation packages
!pip install -q evaluate rouge_score bert_score sacrebleu
!pip install -q sentencepiece protobuf nltk

# Install compatible versions that work with transformers 4.35.2
# !pip install -q peft==0.6.2 bitsandbytes==0.41.3

# Verify installations
import sys
try:
    import transformers
    print(f"✓ transformers: {transformers.__version__}")
except Exception as e:
    print(f"⚠️  transformers: {e}")
    sys.exit(1)

try:
    import peft
    print(f"✓ peft: {peft.__version__}")
except Exception as e:
    print(f"⚠️  peft (for LoRA): {e}")
    print("   → LoRA features will be disabled, but core MoE will work!")

try:
    import evaluate
    print(f"✓ evaluate: installed")
except Exception as e:
    print(f"⚠️  evaluate: {e}")
    print("   → Installing evaluate manually...")
    !pip install -q evaluate
    import evaluate
    print(f"   ✓ evaluate: installed")

print("\n" + "="*70)
print("✅ Installation complete!")
print("="*70)
print("\n⚠️  IMPORTANT: Please RESTART THE KERNEL now!")
print("   📌 Click: 'Kernel' → 'Restart Kernel' (or ↻ button)")
print("   📌 Then run cells starting from the imports cell (skip this cell)")
print("\n💡 Why? This ensures datasets library version 2.14.0 is loaded correctly")
print("💡 Note: The bitsandbytes CUDA warnings above are OK - we'll use CPU quantization")

## 1.2 Import Essential Libraries

Import all required libraries:
- **PyTorch**: For building neural networks
- **Transformers**: For pre-trained models and tokenizers
- **Datasets**: For data loading
- **Numpy & Pandas**: For data manipulation
- **Matplotlib & Seaborn**: For visualization
- **tqdm**: For progress bars

### 🔧 Troubleshooting: If Import Errors Occur

If you see `ImportError: cannot import name 'Cache'` or similar errors, run this cell to fix version conflicts:

In [ ]:
# ⚠️ ONLY RUN THIS CELL IF YOU GET IMPORT ERRORS IN THE NEXT CELL

print("🔧 Attempting to fix import errors...")

# Option 1: Clean install with known working versions (RECOMMENDED)
!pip uninstall -y -q transformers peft bitsandbytes
!pip install -q transformers==4.35.2 peft==0.6.2 bitsandbytes==0.41.3

print("\n✓ Packages reinstalled!")
print("⚠️ IMPORTANT: Click 'Runtime' → 'Restart Runtime' before continuing")
print("   Then skip this cell and run from the imports cell")

# Option 2: If Option 1 doesn't work, try this:
# !pip install -q --force-reinstall --no-deps peft==0.6.2
# print("✓ Force reinstalled peft. Restart runtime and try again.")

# Option 3: Minimal install (MoE without LoRA/Phi-2)
# This works even with version conflicts - disables LoRA features only
# Just proceed with the notebook, LoRA sections will be skipped automatically

### 🔧 Fix Accelerate Version Compatibility

If you see `TypeError: Accelerator.__init__() got an unexpected keyword argument 'dispatch_batches'`, run this cell to upgrade accelerate:

In [ ]:
# Fix accelerate version compatibility issue
print("🔧 Upgrading accelerate library...")

!pip install -q --upgrade accelerate

print("\n✓ Accelerate upgraded successfully!")
print("⚠️ IMPORTANT: Please RESTART THE KERNEL now!")
print("   📌 Click: 'Kernel' → 'Restart Kernel' (or ↻ button)")
print("   📌 Then continue running cells from the imports section")
print("\n💡 This fixes the 'dispatch_batches' error in Seq2SeqTrainer")

In [ ]:
# PyTorch imports
# Suppress warnings and setup environment first
import warnings
import os
import sys
import logging

# Suppress all warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow/CUDA warnings
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Avoid tokenizer warnings

# Suppress absl logging
logging.getLogger('absl').setLevel(logging.ERROR)

# Redirect stderr temporarily to suppress protobuf warnings
import contextlib
stderr_backup = sys.stderr
sys.stderr = open(os.devnull, 'w')

# Core PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Transformers imports (with error handling)
try:
    from transformers import (
        AutoTokenizer,
        AutoModelForSeq2SeqLM,
        AutoModelForCausalLM,
        T5ForConditionalGeneration,
        T5Tokenizer,
        Trainer,
        TrainingArguments,
        Seq2SeqTrainer,
        Seq2SeqTrainingArguments,
        DataCollatorForSeq2Seq,
        DataCollatorForLanguageModeling,
        BitsAndBytesConfig,
        get_linear_schedule_with_warmup
    )
    print("✓ Transformers imported successfully")
except ImportError as e:
    print(f"❌ Import Error: {e}")
    print("💡 Please run the troubleshooting cell above to fix version conflicts")
    raise

# Dataset and evaluation imports
from datasets import load_dataset
import evaluate

# Restore stderr after imports
sys.stderr = stderr_backup

# LoRA/PEFT imports (with error handling)
PEFT_AVAILABLE = False
try:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    PEFT_AVAILABLE = True
    print("✓ PEFT imported successfully")
except ImportError as e:
    print(f"⚠️ PEFT not available: LoRA features will be skipped")
    # Create dummy classes so code doesn't break
    class LoraConfig: pass
    class DummyPeft: pass
    get_peft_model = lambda m, c: m
    prepare_model_for_kbit_training = lambda m: m

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from tqdm.auto import tqdm
import json
import random
import string
from collections import Counter
import gc
import hashlib

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Check GPU availability
if torch.cuda.is_available():
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected - training will be slow")

print("✓ All libraries imported successfully!")
print("✓ Ready to start the assignment!")

### 📝 Note on Package Versions

**Working Configuration for Kaggle/Colab:**
- `transformers==4.35.2` (pre-installed in Kaggle)
- `peft==0.6.2` (compatible with above)
- `bitsandbytes==0.41.3` (for quantization)

**Known Issues:**
- `summac` has strict version requirements - we'll use an alternative consistency checker
- Some dependency warnings are normal and won't affect functionality
- LoRA features (Phi-2) are optional - core MoE works without them

In [ ]:
# Alternative Consistency Checker (replaces summac)
# This is a simple NLI-based consistency checker that doesn't require summac

print("Setting up consistency checker...")

# Try to use summac if available, otherwise use simple alternative
try:
    from summac.model_summac import SummaCZS
    consistency_checker = SummaCZS(granularity="sentence", model_name="vitc")
    print("✓ Using SummaC for consistency checking")
    USE_SUMMAC = True
except:
    print("⚠️  SummaC not available (version conflicts)")
    print("✓ Using simple consistency checker instead")
    USE_SUMMAC = False
    
    # Simple consistency checker using sentence embeddings
    try:
        from sentence_transformers import SentenceTransformer
        consistency_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        def simple_consistency_check(document, summary):
            """
            Simple consistency check using cosine similarity.
            Returns score 0-1 (higher = more consistent)
            """
            try:
                doc_embedding = consistency_model.encode(document, convert_to_tensor=True)
                sum_embedding = consistency_model.encode(summary, convert_to_tensor=True)
                similarity = torch.nn.functional.cosine_similarity(
                    doc_embedding.unsqueeze(0), 
                    sum_embedding.unsqueeze(0)
                )
                return float(similarity.item())
            except:
                return 0.5  # Default score if error
        
        print("  → Using sentence-transformers for consistency")
    except:
        print("  → Using basic heuristic for consistency")
        
        def simple_consistency_check(document, summary):
            """
            Fallback: Check word overlap as consistency proxy.
            """
            doc_words = set(document.lower().split())
            sum_words = set(summary.lower().split())
            if len(sum_words) == 0:
                return 0.0
            overlap = len(doc_words.intersection(sum_words)) / len(sum_words)
            return min(overlap, 1.0)

print("✓ Consistency checker ready")

In [ ]:
# Check device availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

if torch.cuda.is_available():
    print(f"\nGPU Information:")
    print(f"  - GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"  - GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  - CUDA Version: {torch.version.cuda}")
    print(f"  - Number of GPUs: {torch.cuda.device_count()}")
else:
    print("\n⚠️ WARNING: No GPU detected! Training will be very slow.")
    print("Please enable GPU in Kaggle notebook settings.")

In [ ]:
def set_seed(seed=42):
    """
    Set random seeds for reproducibility
    
    Args:
        seed (int): Random seed value
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Additional settings for reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"✓ Random seed set to {seed} for reproducibility")

# Set seed
set_seed(42)

In [ ]:
def clear_memory():
    """
    Clear GPU cache and run garbage collection
    Use this between training different models to free up memory
    """
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()
    print("✓ Memory cleared")

def print_gpu_memory():
    """
    Print current GPU memory usage
    """
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"GPU Memory - Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB")
    else:
        print("No GPU available")

# Test the functions
print_gpu_memory()

---
# 2. Data Loading and Exploration

Load the XSum dataset from Hugging Face and explore its structure and statistics.

## 2.1 Load XSum Dataset from Hugging Face

The XSum (Extreme Summarization) dataset contains:
- **204,045** training samples
- **11,332** validation samples  
- **11,334** test samples
- BBC news articles from 2010-2017
- Task: Generate one-sentence summaries

This cell will download and cache the dataset (may take a few minutes on first run).

In [ ]:
# Load XSum dataset from Hugging Face
from datasets import load_dataset

print("Loading XSum dataset from Hugging Face...")
print("(This will download ~255MB of data on first run)\n")

# Load the dataset - the installation cell already installed datasets 2.14.0
# which should work fine
dataset = load_dataset("EdinburghNLP/xsum")

print("✓ Dataset loaded successfully!")

# ⚡ SPEED OPTIMIZATION: Reduce validation and test set sizes
# This significantly speeds up validation during training and final evaluation
print("\n⚡ Reducing validation and test set sizes for faster training and evaluation...")

original_val_size = len(dataset['validation'])
original_test_size = len(dataset['test'])

dataset['validation'] = dataset['validation'].select(range(500))
dataset['test'] = dataset['test'].select(range(500))

print(f"✓ Validation set reduced: {original_val_size:,} → {len(dataset['validation']):,} samples")
print(f"   Speed improvement: ~{original_val_size/500:.1f}x faster (from ~30 mins to ~1-2 mins)")
print(f"✓ Test set reduced: {original_test_size:,} → {len(dataset['test']):,} samples")
print(f"   Speed improvement: ~{original_test_size/500:.1f}x faster evaluation")

# Display dataset structure
print("\n" + "="*60)
print("Dataset Information")
print("="*60)
print(f"\nDataset splits: {list(dataset.keys())}")
print(f"\nTrain samples: {len(dataset['train']):,}")
print(f"Validation samples: {len(dataset['validation']):,} (reduced for speed ⚡)")
print(f"Test samples: {len(dataset['test']):,} (reduced for speed ⚡)")
print(f"Total samples: {len(dataset['train']) + len(dataset['validation']) + len(dataset['test']):,}")

# Display column names
print(f"\nColumn names: {dataset['train'].column_names}")
print(f"Features: {dataset['train'].features}")

print("\n✓ Dataset loaded successfully!")

## 2.2 Display Sample Data

Examine a few examples from the training set to understand the data format and the summarization task.

In [ ]:
# Display sample examples from the training set
print("="*80)
print("SAMPLE EXAMPLES FROM TRAINING SET")
print("="*80)

# Show 3 random samples
import random
random.seed(42)
sample_indices = random.sample(range(len(dataset['train'])), 3)

for i, idx in enumerate(sample_indices, 1):
    sample = dataset['train'][idx]
    
    print(f"\n{'='*80}")
    print(f"EXAMPLE {i}")
    print(f"{'='*80}\n")
    
    print(f"DOCUMENT (first 500 characters):")
    print("-" * 80)
    print(sample['document'][:500] + "..." if len(sample['document']) > 500 else sample['document'])
    
    print(f"\n{'='*40}")
    print(f"REFERENCE SUMMARY:")
    print("-" * 80)
    print(sample['summary'])
    print()

# Show document and summary word counts for these examples
print("\n" + "="*80)
print("SAMPLE STATISTICS")
print("="*80)
for i, idx in enumerate(sample_indices, 1):
    sample = dataset['train'][idx]
    doc_words = len(sample['document'].split())
    sum_words = len(sample['summary'].split())
    compression_ratio = sum_words / doc_words if doc_words > 0 else 0
    
    print(f"\nExample {i}:")
    print(f"  Document length: {doc_words} words")
    print(f"  Summary length: {sum_words} words")
    print(f"  Compression ratio: {compression_ratio:.3f} ({compression_ratio*100:.1f}%)")

## 2.3 Compute Dataset Statistics

Calculate and visualize statistics about document and summary lengths:
- Mean and standard deviation of document lengths
- Mean and standard deviation of summary lengths  
- Distribution histograms
- Compression ratios

In [ ]:
# Compute comprehensive dataset statistics
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def compute_dataset_statistics(dataset_split, split_name="Dataset"):
    """
    Compute and display statistics for a dataset split
    """
    print(f"\n{'='*80}")
    print(f"{split_name.upper()} STATISTICS")
    print(f"{'='*80}\n")
    
    # Compute lengths
    doc_lengths = [len(x['document'].split()) for x in dataset_split]
    sum_lengths = [len(x['summary'].split()) for x in dataset_split]
    compression_ratios = [s/d if d > 0 else 0 for s, d in zip(sum_lengths, doc_lengths)]
    
    # Document statistics
    print("Document Lengths (in words):")
    print(f"  Mean: {np.mean(doc_lengths):.1f}")
    print(f"  Median: {np.median(doc_lengths):.1f}")
    print(f"  Std Dev: {np.std(doc_lengths):.1f}")
    print(f"  Min: {np.min(doc_lengths)}")
    print(f"  Max: {np.max(doc_lengths)}")
    print(f"  25th percentile: {np.percentile(doc_lengths, 25):.1f}")
    print(f"  75th percentile: {np.percentile(doc_lengths, 75):.1f}")
    
    # Summary statistics
    print("\nSummary Lengths (in words):")
    print(f"  Mean: {np.mean(sum_lengths):.1f}")
    print(f"  Median: {np.median(sum_lengths):.1f}")
    print(f"  Std Dev: {np.std(sum_lengths):.1f}")
    print(f"  Min: {np.min(sum_lengths)}")
    print(f"  Max: {np.max(sum_lengths)}")
    print(f"  25th percentile: {np.percentile(sum_lengths, 25):.1f}")
    print(f"  75th percentile: {np.percentile(sum_lengths, 75):.1f}")
    
    # Compression ratios
    print("\nCompression Ratios:")
    print(f"  Mean: {np.mean(compression_ratios):.4f} ({np.mean(compression_ratios)*100:.2f}%)")
    print(f"  Median: {np.median(compression_ratios):.4f} ({np.median(compression_ratios)*100:.2f}%)")
    print(f"  Std Dev: {np.std(compression_ratios):.4f}")
    
    return doc_lengths, sum_lengths, compression_ratios

# Compute statistics for train set (use a sample for speed)
print("Computing statistics on training set...")
train_sample_size = min(10000, len(dataset['train']))  # Use 10k samples for speed
train_sample = dataset['train'].select(range(train_sample_size))
doc_lengths, sum_lengths, compression_ratios = compute_dataset_statistics(
    train_sample, 
    f"Train (Sample of {train_sample_size:,})"
)

# Create visualizations
print("\nGenerating visualizations...")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('XSum Dataset Statistics', fontsize=16, fontweight='bold')

# Plot 1: Document length distribution
axes[0, 0].hist(doc_lengths, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].axvline(np.mean(doc_lengths), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(doc_lengths):.1f}')
axes[0, 0].axvline(np.median(doc_lengths), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(doc_lengths):.1f}')
axes[0, 0].set_xlabel('Document Length (words)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Document Length Distribution', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Summary length distribution
axes[0, 1].hist(sum_lengths, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].axvline(np.mean(sum_lengths), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(sum_lengths):.1f}')
axes[0, 1].axvline(np.median(sum_lengths), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(sum_lengths):.1f}')
axes[0, 1].set_xlabel('Summary Length (words)', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Summary Length Distribution', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Compression ratio distribution
axes[0, 2].hist(compression_ratios, bins=50, edgecolor='black', alpha=0.7, color='mediumseagreen')
axes[0, 2].axvline(np.mean(compression_ratios), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(compression_ratios):.3f}')
axes[0, 2].set_xlabel('Compression Ratio', fontsize=11)
axes[0, 2].set_ylabel('Frequency', fontsize=11)
axes[0, 2].set_title('Compression Ratio Distribution', fontsize=12, fontweight='bold')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Document length box plot
axes[1, 0].boxplot(doc_lengths, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='steelblue'),
                    medianprops=dict(color='red', linewidth=2))
axes[1, 0].set_ylabel('Document Length (words)', fontsize=11)
axes[1, 0].set_title('Document Length Box Plot', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 5: Summary length box plot
axes[1, 1].boxplot(sum_lengths, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightsalmon', color='coral'),
                    medianprops=dict(color='red', linewidth=2))
axes[1, 1].set_ylabel('Summary Length (words)', fontsize=11)
axes[1, 1].set_title('Summary Length Box Plot', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Plot 6: Scatter plot - Document vs Summary length
sample_indices = np.random.choice(len(doc_lengths), min(1000, len(doc_lengths)), replace=False)
axes[1, 2].scatter([doc_lengths[i] for i in sample_indices], 
                    [sum_lengths[i] for i in sample_indices],
                    alpha=0.5, s=20, color='purple')
axes[1, 2].set_xlabel('Document Length (words)', fontsize=11)
axes[1, 2].set_ylabel('Summary Length (words)', fontsize=11)
axes[1, 2].set_title('Document vs Summary Length (1k sample)', fontsize=12, fontweight='bold')
axes[1, 2].grid(True, alpha=0.3)

# Add trend line
z = np.polyfit([doc_lengths[i] for i in sample_indices], 
               [sum_lengths[i] for i in sample_indices], 1)
p = np.poly1d(z)
x_line = np.linspace(min([doc_lengths[i] for i in sample_indices]), 
                     max([doc_lengths[i] for i in sample_indices]), 100)
axes[1, 2].plot(x_line, p(x_line), "r--", linewidth=2, alpha=0.8, label='Trend')
axes[1, 2].legend()

plt.tight_layout()
plt.savefig('xsum_dataset_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Statistics computed and visualizations created!")
print("✓ Figure saved as 'xsum_dataset_statistics.png'")

---
# Baseline Model 2: Fine-tune FLAN-T5-base

Fine-tune FLAN-T5-base (encoder-decoder model) on XSum dataset.

## Load FLAN-T5-base Model

We'll use **FLAN-T5-base** (~250M parameters):
- Instruction-tuned version of T5
- Encoder-decoder architecture
- Requires "summarize: " prefix for inputs
- Efficient for fine-tuning on summarization tasks

In [ ]:
# Load FLAN-T5-base model and tokenizer with LoRA
from transformers import T5ForConditionalGeneration, T5Tokenizer
from peft import LoraConfig, get_peft_model, TaskType

encoder_decoder_model_name = "google/flan-t5-base"

print(f"Loading model: {encoder_decoder_model_name}")
ed_tokenizer = T5Tokenizer.from_pretrained(encoder_decoder_model_name)
ed_model = T5ForConditionalGeneration.from_pretrained(encoder_decoder_model_name)

# Enable gradient checkpointing to save memory
ed_model.gradient_checkpointing_enable()
print("✓ Gradient checkpointing enabled (saves GPU memory)")

# Move to device
ed_model = ed_model.to(device)

print("\n" + "="*80)
print("CONFIGURING LoRA FOR FLAN-T5")
print("="*80)

# Configure LoRA for T5
lora_config_t5 = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,  # Alpha parameter
    target_modules=["q", "v"],  # T5 uses q, k, v, o for attention (targeting q and v is common)
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM  # T5 is a sequence-to-sequence model
)

# Apply LoRA to model
ed_model = get_peft_model(ed_model, lora_config_t5)

# Print trainable parameters
ed_model.print_trainable_parameters()

# Model information
total_params = sum(p.numel() for p in ed_model.parameters())
trainable_params = sum(p.numel() for p in ed_model.parameters() if p.requires_grad)

print(f"\n{'='*60}")
print(f"FLAN-T5 Model Information (with LoRA)")
print(f"{'='*60}")
print(f"Model name: {encoder_decoder_model_name}")
print(f"Total parameters: {total_params / 1e6:.2f}M")
print(f"Trainable parameters: {trainable_params / 1e6:.2f}M")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")
print(f"Vocabulary size: {len(ed_tokenizer)}")
print(f"Device: {device}")
print(f"\nLoRA Configuration:")
print(f"  Rank (r): {lora_config_t5.r}")
print(f"  Alpha: {lora_config_t5.lora_alpha}")
print(f"  Target modules: {lora_config_t5.target_modules}")
print(f"  Dropout: {lora_config_t5.lora_dropout}")
print(f"\n✓ FLAN-T5 model with LoRA loaded successfully!")
print(f"✓ Only ~{trainable_params / 1e6:.1f}M parameters will be trained (vs {total_params / 1e6:.1f}M full fine-tuning)")

## 4.2 Preprocess Dataset for FLAN-T5

Create preprocessing function to:
- Add "summarize: " prefix (required for T5 models)
- Tokenize documents (max_length=512)
- Tokenize summaries (max_length=64)
- Create labels for training

Apply to train, validation, and test splits.

In [ ]:
# Preprocessing function for FLAN-T5
def preprocess_function_t5(examples):
    """
    Preprocess data for FLAN-T5
    Requires "summarize: " prefix for T5 models
    """
    # Add T5 prefix to documents
    inputs = ["summarize: " + doc for doc in examples["document"]]
    
    # Tokenize documents
    model_inputs = ed_tokenizer(
        inputs, 
        max_length=512, 
        truncation=True,
        padding=False  # Will be handled by DataCollator
    )
    
    # Tokenize summaries (targets)
    labels = ed_tokenizer(
        text_target=examples["summary"],
        max_length=64,
        truncation=True,
        padding=False
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Preprocessing datasets for FLAN-T5...")
print("This may take a few minutes...")

# Tokenize train dataset
print("\nProcessing train set...")
train_dataset = dataset["train"].map(
    preprocess_function_t5,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing train set"
)

# Tokenize validation dataset
print("Processing validation set...")
val_dataset = dataset["validation"].map(
    preprocess_function_t5,
    batched=True,
    remove_columns=dataset["validation"].column_names,
    desc="Tokenizing validation set"
)

# Tokenize test dataset
print("Processing test set...")
test_dataset_processed = dataset["test"].map(
    preprocess_function_t5,
    batched=True,
    remove_columns=dataset["test"].column_names,
    desc="Tokenizing test set"
)

print(f"\n✓ Preprocessing complete!")
print(f"Train samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Test samples: {len(test_dataset_processed):,}")

## 4.3 Setup Training Arguments

Configure training parameters for FLAN-T5-base with LoRA **optimized for SPEED**:
- Learning rate: 3e-4 (higher for LoRA)
- Batch size: 8 per device (optimal for T4 GPU speed)
- NO gradient accumulation (direct weight updates)
- Epochs: 1 (testing, increase if needed)
- Mixed precision training (fp16)
- **Gradient checkpointing DISABLED** (trades memory for speed)
- Parallel data loading enabled
- Less frequent evaluation and saving

**Speed optimizations:**
- Disabled gradient checkpointing (faster forward/backward pass)
- Added dataloader workers for parallel data loading
- Reduced evaluation frequency (1000 steps vs 500)
- Only 1 epoch initially to validate speed

In [ ]:
# Setup training arguments for FLAN-T5 with LoRA
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./flan_t5_xsum_lora",  # Changed to reflect LoRA training
    eval_strategy="steps",  # Changed from evaluation_strategy in latest transformers
    eval_steps=1000,  # Less frequent eval (was 500)
    learning_rate=3e-4,  # Higher learning rate for LoRA (was 5e-5)
    per_device_train_batch_size=8,  # Balanced for T4 GPU speed
    per_device_eval_batch_size=8,   # Balanced for T4 GPU
    gradient_accumulation_steps=1,  # NO accumulation - direct updates
    num_train_epochs=3,  # Train for 3 epochs
    weight_decay=0.01,
    save_total_limit=2,  # Save fewer checkpoints
    save_steps=2000,  # Less frequent saves
    logging_steps=50,  # More frequent logging to see progress
    predict_with_generate=True,
    generation_max_length=64,
    fp16=True,  # Mixed precision training
    gradient_checkpointing=False,  # DISABLE - it slows down training!
    push_to_hub=False,
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,
    report_to="none",
    warmup_steps=500,
    dataloader_num_workers=2,  # Parallel data loading
    dataloader_pin_memory=True,  # Faster data transfer to GPU
)

print("="*60)
print("TRAINING CONFIGURATION (LoRA - SPEED OPTIMIZED)")
print("="*60)
print(f"Output directory: {training_args.output_dir}")
print(f"Learning rate: {training_args.learning_rate} (higher for LoRA)")
print(f"Batch size per device: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Number of epochs: {training_args.num_train_epochs}")
print(f"Mixed precision (fp16): {training_args.fp16}")
print(f"Gradient checkpointing: {training_args.gradient_checkpointing} (DISABLED for speed)")
print(f"Evaluation strategy: {training_args.eval_strategy}")
print(f"Save steps: {training_args.save_steps}")

# Calculate approximate training time
total_steps = (len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)) * training_args.num_train_epochs
print(f"\nTotal training steps: ~{total_steps:,}")
print(f"Estimated training time: 20-30 minutes for 1 epoch (T4 GPU)")
print(f"\n💡 Speed Optimizations Applied:")
print(f"   - Gradient checkpointing OFF (faster but uses more memory)")
print(f"   - Dataloader workers: 2 (parallel data loading)")
print(f"   - Batch size: 8 (optimal for T4 speed)")
print(f"   - Only {trainable_params / 1e6:.1f}M trainable params with LoRA")
print(f"   - 1 epoch for initial testing")
print("\n✓ Training arguments configured!")

## 4.4 Define Compute Metrics Function

Create function to compute ROUGE scores during training for model selection.

In [ ]:
# Define compute metrics function for evaluation
import evaluate

# Load ROUGE metric
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    """
    Compute ROUGE scores for evaluation
    
    Args:
        eval_pred: Tuple of (predictions, labels)
    
    Returns:
        Dictionary of ROUGE scores
    """
    predictions, labels = eval_pred
    
    # Clip predictions to valid token ID range to avoid IndexError
    vocab_size = len(ed_tokenizer)
    predictions = np.clip(predictions, 0, vocab_size - 1)
    
    # Decode predictions
    decoded_preds = ed_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in labels (padding token)
    labels = np.where(labels != -100, labels, ed_tokenizer.pad_token_id)
    decoded_labels = ed_tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Compute ROUGE scores
    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    
    # Return as percentages
    return {k: round(v * 100, 2) for k, v in result.items()}

print("✓ Compute metrics function defined!")
print("  Metrics: ROUGE-1, ROUGE-2, ROUGE-L, ROUGE-Lsum")
print("  ⚠️  Token ID clipping enabled to prevent IndexError")

## 4.5 Create Trainer and Start Training

Initialize Seq2SeqTrainer with:
- Model and tokenizer
- Training arguments
- Train and validation datasets
- Data collator
- Metrics function

Start training (will take 4-6 hours).

In [ ]:
# Create Seq2SeqTrainer and start training
from transformers import DataCollatorForSeq2Seq

# Clear any leftover memory from previous models
print("Clearing GPU memory before training...")
clear_memory()

print("="*80)
print("INITIALIZING FLAN-T5 TRAINER")
print("="*80)

# Create data collator
data_collator = DataCollatorForSeq2Seq(
    ed_tokenizer, 
    model=ed_model,
    padding=True
)

# Create trainer
trainer = Seq2SeqTrainer(
    model=ed_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=ed_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n✓ Trainer initialized!")
print(f"Training dataset size: {len(train_dataset):,}")
print(f"Validation dataset size: {len(val_dataset):,}")
print(f"\n💡 Speed-optimized LoRA configuration:")
print(f"   - Only {trainable_params:,} trainable parameters (0.94% of model)")
print(f"   - Batch size: 8 (optimal for T4)")
print(f"   - NO gradient accumulation (immediate updates)")
print(f"   - Gradient checkpointing OFF (faster training)")
print(f"   - FP16 mixed precision + parallel data loading")
print(f"   - Higher learning rate (3e-4 for LoRA)")

# Start training
print(f"\n{'='*80}")
print("STARTING FLAN-T5 LoRA TRAINING (SPEED-OPTIMIZED)")
print(f"{'='*80}\n")
print("⏰ Expected time: 20-30 minutes for 1 epoch...")
print("💡 Training progress will be logged every 50 steps")
print("📊 Evaluation will run every 1000 steps")
print("🚀 If speed looks good, increase to 3 epochs for final training\n")

# Train the model
trainer.train()

print(f"\n{'='*80}")
print("TRAINING COMPLETE!")
print(f"{'='*80}")

# Save the final model (LoRA adapters + tokenizer)
ed_model.save_pretrained("./flan_t5_xsum_lora_final")
ed_tokenizer.save_pretrained("./flan_t5_xsum_lora_final")

print("\n✓ LoRA adapters saved to './flan_t5_xsum_lora_final'")
print("✓ Training complete!")
print("\n💡 To load this model later:")
print("   from peft import PeftModel")
print("   base_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base')")
print("   model = PeftModel.from_pretrained(base_model, './flan_t5_xsum_lora_final')")

## 4.6 Run Inference on Test Set

Generate summaries using the fine-tuned model on the test set.

In [ ]:
# Run inference on test set with fine-tuned FLAN-T5
def generate_summaries_finetuned(model, tokenizer, test_data_raw, batch_size=8):
    """
    Generate summaries with fine-tuned FLAN-T5 model
    
    Args:
        model: Fine-tuned model
        tokenizer: Tokenizer
        test_data_raw: Raw test dataset
        batch_size: Batch size for inference
    
    Returns:
        predictions: List of generated summaries
    """
    model.eval()
    predictions = []
    
    print(f"Generating summaries with batch size {batch_size}...")
    
    for i in tqdm(range(0, len(test_data_raw), batch_size), desc="FLAN-T5 Inference"):
        batch = test_data_raw[i:i+batch_size]
        
        # Add "summarize: " prefix for T5
        inputs = tokenizer(
            batch["document"],
            max_length=512,
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=64,
                num_beams=4,
                length_penalty=2.0,
                early_stopping=True,
                no_repeat_ngram_size=3
            )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded)
    
    return predictions

print("="*80)
print("RUNNING INFERENCE WITH FINE-TUNED FLAN-T5")
print("="*80)

# Move model to GPU
ed_model.to(device)

# Use same test subset as BART for comparison
if use_full_test:
    test_for_inference = dataset["test"]
else:
    test_for_inference = dataset["test"].select(range(1000))

print(f"\nGenerating summaries for {len(test_for_inference):,} samples...")

# Generate summaries
ed_predictions = generate_summaries_finetuned(
    ed_model,
    ed_tokenizer,
    test_for_inference,
    batch_size=8
)

print(f"\n✓ Generated {len(ed_predictions):,} summaries!")

# Save results
ed_results_df = pd.DataFrame({
    'document': test_for_inference["document"],
    'reference': test_for_inference["summary"],
    'prediction': ed_predictions
})

ed_results_df.to_csv('flan_t5_predictions.csv', index=False)
print(f"✓ Results saved to 'flan_t5_predictions.csv'")

# Show some examples
print(f"\n{'='*80}")
print("SAMPLE PREDICTIONS")
print(f"{'='*80}")

for i in range(3):
    print(f"\n{'-'*80}")
    print(f"EXAMPLE {i+1}")
    print(f"{'-'*80}")
    print(f"\nDocument (first 200 chars):")
    print(test_for_inference[i]['document'][:200] + "...")
    print(f"\nReference: {test_for_inference[i]['summary']}")
    print(f"\nPrediction: {ed_predictions[i]}")

# Clear GPU memory
del ed_model
clear_memory()
print("\n✓ FLAN-T5 inference complete and memory cleared!")